In [1]:
# Ensure datasets library is installed and load the dataset as in EDA notebook
!pip install datasets
from datasets import load_dataset

dataset = load_dataset("millat/e-commerce-orders")
df = dataset['train'].to_pandas()
df.head()

c:\Users\UsEr\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,order_id,customer_id,product_id,category,price,quantity,order_date,shipping_date,delivery_status,payment_method,device_type,channel,shipping_address,billing_address,customer_segment
0,b8ec9f86-5919-4b71-a5f7-945e7c0a3db0,2031,845,Books,45.95,4,2024-04-20 14:59:58.897063,2024-04-27 14:59:58.897063,Shipped,PayPal,Mobile,Paid Search,"976 Kevin Station, Davidmouth, Hawaii 92563","38182 Ariel Expressway, Campbellland, Oklahoma...",VIP
1,5ea92c47-c5b2-4bdd-8a50-d77efd77ec89,2350,995,Electronics,403.17,3,2024-04-20 14:59:58.897063,2024-04-22 14:59:58.897063,Delivered,PayPal,Mobile,Paid Search,"72166 Cunningham Crescent, East Nicholasside, ...","38199 Edwin Plain, Johnborough, Maine 81826",Returning
2,5cc48ce0-2c6d-4448-af3f-96f8a910d45b,1818,997,Beauty,317.45,2,2024-04-20 14:59:58.897063,2024-04-27 14:59:58.897063,Shipped,Credit Card,Mobile,Email,"2446 Johnson Junctions, Lynchtown, Minnesota 7...","58086 Smith Stream Suite 994, Lake Pamelabury,...",Returning
3,74d5c0f4-53f0-4367-a5c5-baaa114c2d9f,472,385,Home,24.08,3,2024-04-20 14:59:58.897063,2024-04-24 14:59:58.897063,Shipped,PayPal,Tablet,Social,"3113 Jessica Knolls, North Joshuafort, Alabama...","484 Palmer Harbors Apt. 866, Dustintown, Nebra...",VIP
4,7a630323-8ac8-406e-875a-4bcdead440ab,1075,31,Clothing,494.90,1,2024-04-20 14:59:58.897063,2024-04-25 14:59:58.897063,Delivered,PayPal,Tablet,Organic,"58196 Burgess Heights Suite 315, Douglasland, ...","67094 Schaefer Villages Suite 369, Douglasches...",VIP


# 4. MLOps Deployment: MLflow, GCS, and Vertex AI
This notebook documents the transfer of model artifacts to Google Cloud Storage and the deployment of the model as a live endpoint on Vertex AI.

## 1. Import Required Libraries
We import libraries for model packaging, cloud storage, and deployment.

In [2]:
import joblib
import os
# For Google Cloud Storage and Vertex AI, use the google-cloud-storage and google-cloud-aiplatform packages
# !pip install google-cloud-storage google-cloud-aiplatform
from google.cloud import storage
from google.cloud import aiplatform

## 2. Package and Save the Final Model
We combine the preprocessing pipeline and the best classification model into a single pipeline and save it as model.joblib.

In [7]:
# Load the best model pipeline from previous training (if not already in memory)
model_path = '../artifacts/model.joblib'

if os.path.exists(model_path):
	cls_pipeline = joblib.load(model_path)
else:
	# Example: create a dummy pipeline for demonstration (replace with your actual pipeline)
	from sklearn.pipeline import Pipeline
	from sklearn.preprocessing import StandardScaler
	from sklearn.ensemble import RandomForestClassifier

	# Assume 'customer_segment' is the target
	X = df.drop(columns=['customer_segment'])
	y = df['customer_segment']

	# For demonstration, use only numeric columns
	numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
	X_numeric = X[numeric_cols]

	cls_pipeline = Pipeline([
		('scaler', StandardScaler()),
		('clf', RandomForestClassifier(n_estimators=10, random_state=42))
	])
	cls_pipeline.fit(X_numeric, y)

# Save the model pipeline (optional, if you want to overwrite)
os.makedirs('../artifacts', exist_ok=True)
joblib.dump(cls_pipeline, model_path)
print('Model pipeline saved to ../artifacts/model.joblib')

Model pipeline saved to ../artifacts/model.joblib


## 3. Prepare requirements.txt
We list all required Python packages for deployment.

## 4. Upload Artifacts to Google Cloud Storage (GCS)
We upload model.joblib and requirements.txt to a GCS bucket for deployment.

In [ ]:
# Set your GCS bucket name
bucket_name = 'your-gcs-bucket-name'

# Initialize GCS client
client = storage.Client()
bucket = client.bucket(bucket_name)

# Upload model.joblib
model_blob = bucket.blob('model.joblib')
model_blob.upload_from_filename('../artifacts/model.joblib')
print('Uploaded model.joblib to GCS')

# Upload requirements.txt
req_blob = bucket.blob('requirements.txt')
req_blob.upload_from_filename('../artifacts/requirements.txt')
print('Uploaded requirements.txt to GCS')

## 5. Deploy Model to Vertex AI Endpoint
We import the model from GCS, select the pre-built Scikit-learn container, and deploy as a live endpoint.

In [ ]:
# Set your project and region
project = 'your-gcp-project-id'
location = 'us-central1'

# Initialize Vertex AI
aiplatform.init(project=project, location=location)

# Import model from GCS
model = aiplatform.Model.upload(
    display_name='AuraCart Customer Segment Model',
    artifact_uri=f'gs://{bucket_name}/model.joblib',
    serving_container_image_uri='us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest',
    requirements=['gs://{}/requirements.txt'.format(bucket_name)]
)

# Deploy to endpoint
endpoint = model.deploy(machine_type='n1-standard-2')
print('Model deployed to Vertex AI endpoint:', endpoint.resource_name)

## 6. Test the Live Endpoint
Send a sample JSON payload to the endpoint and display the prediction response.

In [ ]:
# Example: send a test request (replace with your endpoint details)
from google.cloud.aiplatform.gapic.prediction_service_client import PredictionServiceClient
from google.cloud.aiplatform.gapic.schema import predict

endpoint_id = 'your-endpoint-id'
instance = {
    # Fill with a sample input matching the model's expected features
}

client = PredictionServiceClient(client_options={"api_endpoint": f"{location}-aiplatform.googleapis.com"})
response = client.predict(
    endpoint=f"projects/{project}/locations/{location}/endpoints/{endpoint_id}",
    instances=[instance],
)
print('Prediction response:', response)